**[🏠 Course Home](00_START_HERE.ipynb)** | **🐍 [Python](../python/01_foundations_and_conjugate_updating.ipynb)** | **🔵 [R](../r/01_foundations_and_conjugate_updating.ipynb)** | **⬅️ [Prev: Ch 00: Course Overview & Roadmap](00_START_HERE.ipynb)** | **➡️ [Next: Ch 02: Continuous Normal Conjugacy](02_normal_conjugate_updating.ipynb)**

*🧭 Part I: Foundations, Conjugacy & Estimating a Mean — Chapter 01 of 22*

---

# Chapter 1: Learning from Evidence — How Beliefs Update
### *The Balance Scale, Laplace's Sunrise, and The Gaussian Tug-of-War*

---

## 1. What Are We Trying to Do?

Imagine you are tracking an unknown rate:
* The conversion rate of a checkout button.
* The probability that a network packet is dropped.
* The failure rate of an automated test.

You start with an initial belief about this rate, and then you start observing trials (successes and failures). 
**How should your belief rationally change with each new observation?**

---

## 2. The Beta-Binomial Model: The Balance Scale of Evidence

The formal mathematical system for updating beliefs about a binary success
rate from observed pass/fail outcomes is known as the **Beta-Binomial conjugate
model**. But you do not need any calculus to understand how it works.

### The Prior as "Prior Observations"
Imagine a physical balance scale with two buckets:
* The **Success Bucket** (labeled $\alpha$)
* The **Failure Bucket** (labeled $\beta$)

Before you collect a single piece of real-world data, you place some initial weights in those buckets representing your **prior experience**:
* If you have no idea what the rate is, you put **1 pebble** in the success bucket and **1 pebble** in the failure bucket ($\alpha = 1, \beta = 1$). This represents total neutrality: every rate between 0% and 100% is equally plausible.
* If you are monitoring a production database where historical telemetry shows roughly 99 successful queries for every 1 timeout, you put **99 pebbles** in the success bucket and **1 pebble** in the failure bucket ($\alpha = 99, \beta = 1$).

### The Data Arrives
Now, you run your experiment and observe $k$ successes and $n - k$ failures.
How does the update happen?

> [!TIP]
> **The Conjugate Magic: Simple Addition**
> 
> You literally just drop the new successes into the success bucket, and the new failures into the failure bucket!
> 
> $$\text{New Successes} = \alpha_{\text{prior}} + k$$
> $$\text{New Failures} = \beta_{\text{prior}} + (n - k)$$

That's it. That is the entire mathematical update!
* If your prior was $\alpha = 1, \beta = 1$ (neutral), and you observe **8 successes** and **2 failures**, your updated belief is simply:
  $$\alpha_{\text{new}} = 1 + 8 = 9, \qquad \beta_{\text{new}} = 1 + 2 = 3$$
* What is your updated estimate now?
  * **Expected Value (Posterior Mean / Center of Gravity)**:
    $$\mathbb{E}[\theta] = \frac{\text{Successes}}{\text{Total Pebbles}} = \frac{\alpha}{\alpha + \beta} = \frac{9}{9 + 3} = \frac{9}{12} = 75.0\%$$
  * **Peak Mode (Maximum A Posteriori / Most Probable Individual Point)**:
    $$\text{Mode} = \frac{\alpha - 1}{\alpha + \beta - 2} = \frac{9 - 1}{9 + 3 - 2} = \frac{8}{10} = 80.0\%$$

<details>
<summary><b>📐 Mathematical Deep Dive: Why is Expected Value Exactly $\frac{\alpha}{\alpha + \beta}$? (The 3-Line Beta Integral)</b></summary>

By definition, the expected value of any continuous random variable with probability density function $p(\theta)$ on $[0, 1]$ is:
$$\mathbb{E}[\theta] = \int_0^1 \theta \cdot p(\theta) \, d\theta$$

For a $\text{Beta}(\alpha, \beta)$ distribution, $p(\theta) = \frac{1}{\text{B}(\alpha, \beta)} \theta^{\alpha - 1} (1 - \theta)^{\beta - 1}$, where $\text{B}(\alpha, \beta) = \frac{\Gamma(\alpha)\Gamma(\beta)}{\Gamma(\alpha + \beta)}$.

1. **Multiplying by $\theta$ absorbs into the exponent**:
   $$\mathbb{E}[\theta] = \frac{1}{\text{B}(\alpha, \beta)} \int_0^1 \theta \cdot \left[ \theta^{\alpha - 1} (1 - \theta)^{\beta - 1} \right] d\theta = \frac{1}{\text{B}(\alpha, \beta)} \int_0^1 \theta^{(\alpha + 1) - 1} (1 - \theta)^{\beta - 1} \, d\theta$$

2. **The integral is simply a new Beta function $\text{B}(\alpha + 1, \beta)$**:
   $$\mathbb{E}[\theta] = \frac{\text{B}(\alpha + 1, \beta)}{\text{B}(\alpha, \beta)} = \frac{\frac{\Gamma(\alpha + 1)\Gamma(\beta)}{\Gamma(\alpha + 1 + \beta)}}{\frac{\Gamma(\alpha)\Gamma(\beta)}{\Gamma(\alpha + \beta)}}$$

3. **Applying the Gamma identity $\Gamma(x + 1) = x \Gamma(x)$**:
   Because $\Gamma(\alpha + 1) = \alpha \Gamma(\alpha)$ and $\Gamma(\alpha + \beta + 1) = (\alpha + \beta) \Gamma(\alpha + \beta)$, the $\Gamma$ terms cancel cleanly:
   $$\mathbb{E}[\theta] = \frac{\frac{\alpha \Gamma(\alpha)\Gamma(\beta)}{(\alpha + \beta)\Gamma(\alpha + \beta)}}{\frac{\Gamma(\alpha)\Gamma(\beta)}{\Gamma(\alpha + \beta)}} = \frac{\alpha}{\alpha + \beta} \qquad \blacksquare$$

In physical terms, the expected value is simply the **center of mass** of the pebble balance scale: the proportion of positive evidence over the total evidence pool.
</details>

<details>
<summary><b>📐 Mathematical Deep Dive: Why is the Peak Mode Exactly $\frac{\alpha - 1}{\alpha + \beta - 2}$? (Calculus Optimization)</b></summary>

While the **Expected Value (Mean)** is the center of mass, the **Mode** is the literal highest peak of the probability density curve—the single value $\theta^*$ maximizing $p(\theta)$.

For $\text{Beta}(\alpha, \beta)$ with $\alpha, \beta > 1$:
$$p(\theta) = \frac{1}{\text{B}(\alpha, \beta)} \theta^{\alpha - 1} (1 - \theta)^{\beta - 1}$$

Because the natural logarithm is strictly increasing, maximizing $p(\theta)$ is identical to maximizing $\ln p(\theta)$:

1. **Take the log-density**:
   $$\ln p(\theta) = -\ln \text{B}(\alpha, \beta) + (\alpha - 1) \ln \theta + (\beta - 1) \ln(1 - \theta)$$

2. **Set the first derivative to zero (find the peak)**:
   $$\frac{d}{d\theta} \ln p(\theta) = \frac{\alpha - 1}{\theta} - \frac{\beta - 1}{1 - \theta} = 0$$

3. **Solve for $\theta$**:
   $$\frac{\alpha - 1}{\theta} = \frac{\beta - 1}{1 - \theta} \implies (\alpha - 1)(1 - \theta) = (\beta - 1)\theta$$
   $$\alpha - 1 = [(\alpha - 1) + (\beta - 1)] \theta = (\alpha + \beta - 2) \theta$$
   $$\theta^* = \text{Mode} = \frac{\alpha - 1}{\alpha + \beta - 2} \qquad (\text{for } \alpha, \beta > 1)$$

*(Checking the second derivative confirms $\frac{d^2}{d\theta^2} \ln p(\theta) = -\frac{\alpha - 1}{\theta^2} - \frac{\beta - 1}{(1 - \theta)^2} < 0$, guaranteeing a unique concave maximum).*

**Why the "$-1$" and "$-2$"?**  
Under a flat, uninformative prior ($\text{Beta}(1, 1)$), the prior adds zero net pull to the peak. When you observe $k = 8$ passes in $n = 10$ trials, the posterior is $\text{Beta}(1+8, 1+2) = \text{Beta}(9, 3)$.  
The Mode subtracts the baseline offset to recover the empirical sample proportion:
$$\text{Mode} = \frac{9 - 1}{(9 - 1) + (3 - 1)} = \frac{8}{8 + 2} = \frac{8}{10} = 80.0\%$$
The Mode recovers the classical Maximum Likelihood Estimate ($\hat{p} = k/n$), while the Mean ($\frac{9}{12} = 75.0\%$) regularizes slightly toward the prior center of gravity.
</details>


```
                         THE BALANCE SCALE OF BELIEF
                         
         Prior Weights                 New Real Data                Updated Posterior
      [alpha=1]   [beta=1]    +     [8 Pass]   [2 Fail]    =     [alpha=9]   [beta=3]
         \             /               \             /              \             /
          \___________/                 \___________/                \___________/
                ▲                             ▲                            ▲
        Neutral State                  Sample Observed               75% Pass Belief
```

Notice what happened:
1. **Your prior acts like prior data**: Having a prior of $\alpha = 10, \beta = 10$ is mathematically identical to having already observed 20 past runs (10 passes, 10 fails).
2. **Small data leaves you uncertain**: If you only observe 2 runs, the prior pebbles still hold significant weight.
3. **Large data overwhelms the prior**: If you observe 10,000 runs, whether you started with 1 pebble or 10 pebbles becomes completely irrelevant—the data drowns out the prior.

---


> 🐍 **See the Code**: Run the streaming Beta update on real CI build batches in Python!  
> Open **[Python Chapter 1: Part 5 — Interactive Streaming Updates](../python/01_foundations_and_conjugate_updating.ipynb#part-5-interactive-streaming-updates-across-ci-runs)**.


---

### Applied Decision: Comparing Two Variants (Bayesian A/B Testing & Relative Uplift)

Once you understand how the balance scale works for a single test, a natural question arises: **How do we compare two competing options?**
* Variant A (Old Checkout / Baseline Architecture): $100$ conversions in $1{,}000$ visitors $\to \text{Beta}(101, 901)$ (Mean $\approx 10.0\%$).
* Variant B (New Checkout / New Retry Strategy): $125$ conversions in $1{,}000$ visitors $\to \text{Beta}(126, 876)$ (Mean $\approx 12.5\%$).

#### Absolute Difference vs. Relative Uplift
* **Absolute Difference**: $\Delta = \theta_B - \theta_A = 12.5\% - 10.0\% = +2.5$ percentage points.
* **Relative Uplift**: $U = \frac{\theta_B - \theta_A}{\theta_A} = \frac{12.5\% - 10.0\%}{10.0\%} = +25.0\%$ percentage gain.

> **Why Relative Uplift Matters**:  
> A $+2.5$ percentage point increase from $1.0\%$ to $3.5\%$ is a **$+250\%$ business explosion** (more than tripling revenue). But from $50.0\%$ to $52.5\%$, it is only a $+5\%$ modest bump. Relative uplift scales with business impact.

#### Why Is It a *Distribution*?
In classical statistics, people calculate a single number ($+25\%$) and struggle to compute confidence intervals for ratios.  
In Bayesian inference, because both $\theta_A$ and $\theta_B$ are full probability curves, **their ratio is also a full probability curve (The Relative Uplift Distribution)**:
1. **Probability of Any Win**: $P(\theta_B > \theta_A) = 96.8\%$. (There is a $96.8\%$ chance B is superior, and only a $3.2\%$ chance of regression).
2. **Expected Percentage Gain**: The average relative uplift is $+25.6\%$.
3. **Plausible Range (95% Credible Interval)**: The true percentage gain lies between **$+0.8\%$ and $+53.2\%$**.

---

## 3. Laplace's Rule of Succession: Will the Sun Rise Tomorrow?

In the late 1700s, the great French mathematician Pierre-Simon Laplace posed a famous philosophical question:
> *"Suppose you have observed the sun rise every single day for the past 5,000 years (roughly 1,826,200 consecutive sunrises, with 0 failures). What is the probability that the sun will rise again tomorrow?"*

A naive frequentist approach says:
$$\hat{p} = \frac{1{,}826{,}200}{1{,}826{,}200} = 1.0 \quad (100\%)$$
A probability of 100% means **absolute, dogmatic certainty**. It means it is physically and logically impossible for the sun not to rise.

Laplace recognized that this is dangerous nonsense. No finite number of observations can ever justify absolute certainty.
Using Bayesian updating with a neutral prior ($\alpha = 1, \beta = 1$):
* You have $1,826,200$ observed successes.
* You add your 1 prior success pebble ($\alpha = 1 + 1,826,200$).
* You add your 1 prior failure pebble ($\beta = 1 + 0 = 1$).

Laplace's formula for the next event is simply:

$$P(\text{Success on Next Trial}) = \frac{k + 1}{n + 2}$$

* For the sun: $\frac{1{,}826{,}201}{1{,}826{,}202} \approx 0.99999945$. Extremely close to 1, but leaving an honest sliver of uncertainty for cosmic catastrophes.
* For a software test: If you run a new integration test **10 times** and it passes all 10 times ($k = 10, n = 10$):
  * Naive calculation: $10/10 = 100\%$ reliable.
  * Laplace's Rule of Succession: $\frac{10 + 1}{10 + 2} = \frac{11}{12} \approx \mathbf{91.7\%}$.
  * Laplace reminds you that after only 10 runs, there is still an **8.3% chance** the next run will fail!

---

## 4. Continuous Measurements: The Gaussian Tug-of-War

What if we are not counting binary successes and failures, but measuring continuous physical quantities—like response latency, temperature, or vehicle speed?

In this case, we use the **Normal (Gaussian) conjugate model**. And its intuition is just as physical: **The Tug-of-War**.

### The Two GPS Sensors
Imagine a self-driving car trying to determine its position on a road:
* **Sensor A (Cheap GPS)**: Reports the car is at position **10.0 meters**, with an error uncertainty of $\pm 4.0$ meters.
* **Sensor B (High-Precision Lidar)**: Reports the car is at position **12.0 meters**, with an error uncertainty of $\pm 1.0$ meter.

Where is the car?
Do you just take the simple average: $(10 + 12) / 2 = 11.0$ meters?
**Of course not!** Sensor B is four times more precise than Sensor A. Sensor B should have much more influence on your final belief.

```
                         THE GAUSSIAN TUG-OF-WAR
                         
           Sensor A                                         Sensor B
       (Weak / Noisy)                                   (Strong / Precise)
        Pos: 10.0m                                         Pos: 12.0m
       Variance: 16.0                                     Variance: 1.0
          (Weight 1)                                       (Weight 16)
              \                                                 /
               \======================▲========================/
                                  Final Position:
                                      11.88m
```

> [!NOTE]
> ### 🪢 Inverse-Variance Weighting
> In Bayesian inference, certainty is the inverse of variance:
> $$\text{Precision} = \frac{1}{\text{Variance}}$$
> 
> When you combine two Gaussian sources of information (whether it's a Prior and Data, or two Sensors):
> 1. **Each source pulls the estimate toward itself with a force equal to its precision.**
> 2. **The final precision is simply the sum of the two precisions!**
> 
> Because Sensor B has 16 times the precision of Sensor A, the final estimate sits at **11.88 meters** (massively pulled toward Sensor B), and the combined uncertainty shrinks below that of either sensor alone!

---


> 🐍 **See the Code**: Simulate sensor fusion and inverse-variance Gaussian updates in Python!  
> Open **[Python Chapter 2: Part 3 — The Three Archetypes of Bayesian Evidence](../python/02_normal_conjugate_updating.ipynb#part-3-the-three-archetypes-of-bayesian-evidence-simulation)**.


---

## 5. What Does "Conjugacy" Actually Mean?

Throughout literature, you will hear mathematicians praise **"Conjugate Priors"**.
What does that word actually mean in plain English?

> [!IMPORTANT]
> **Conjugacy is a Mathematical Gift: Shape Preservation**
> 
> A prior is called "conjugate" to a likelihood if **the updated belief (posterior) has the exact same mathematical shape as the starting belief (prior)**.
> 
> * If you start with a Beta shape and observe Binomial data $\to$ you end up with a Beta shape.
> * If you start with a Bell Curve (Normal) and observe Normal data $\to$ you end up with a Bell Curve.
> * If you start with a Gamma shape and observe Poisson counts $\to$ you end up with a Gamma shape.

### Why is this so desirable?
Because **it turns calculus into simple arithmetic**!
Instead of solving an impossible multidimensional integral, you just add numbers:
* Beta: Add successes and failures.
* Normal: Add weighted means and precisions.

### The Catch
Conjugacy is a luxury that only exists for a tiny handful of textbook mathematical pairs. 
The moment you want to model real-world complexities—like a failure rate that depends on server load, network latency, and time of day—the neat mathematical conjugacy breaks down.

First, in **Chapter 2**, we extend conjugacy to continuous Gaussian measurements—and then in **Chapter 5**, we see what to do when conjugacy fails completely!

---


---
## 🧭 Course Navigation

* 🏠 **[Course Home & Master Syllabus](00_START_HERE.ipynb)**
* ⬅️ **Previous**: **[Ch 00: Course Overview & Roadmap](00_START_HERE.ipynb)** *(Part I: Foundations, Conjugacy & Estimating a Mean)*
* ➡️ **Next**: **[Ch 02: Continuous Normal Conjugacy](02_normal_conjugate_updating.ipynb)** *(Part I: Foundations, Conjugacy & Estimating a Mean)*
* 🔄 **Companion Tracks for Chapter 01**: [📖 Conceptual](../conceptual/01_foundations_and_conjugate_updating.ipynb) | [🐍 Python](../python/01_foundations_and_conjugate_updating.ipynb) | [🔵 R](../r/01_foundations_and_conjugate_updating.ipynb)
